In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

emp_data = [
    Row(emp_id=1, emp_name="Alice", emp_salary=50000,emp_dept_id=101, emp_location="New York"),
    Row(emp_id=2, emp_name="Bob", emp_salary=60000,emp_dept_id=102, emp_location="Los Angeles"),
    Row(emp_id=3, emp_name="Charlie", emp_salary=55000,emp_dept_id=101, emp_location="Chicago"),
    Row(emp_id=4, emp_name="David", emp_salary=70000,emp_dept_id=103, emp_location="San Francisco"),
    Row(emp_id=5, emp_name="Eve", emp_salary=48000,emp_dept_id=102, emp_location="Houston")
]

dept_data = [
    Row(dept_id=101, dept_name="Engineering", dept_head="John",dept_location="New York"),
    Row(dept_id=102, dept_name="Marketing", dept_head="Mary",dept_location="Los Angeles"),
    Row(dept_id=103, dept_name="Finance", dept_head="Frank",dept_location="Chicago"),
    Row(dept_id=104, dept_name="HR", dept_head="Sham",dept_location="India"),
]

emp_columns = ["emp_id", "emp_name", "emp_salary", "emp_dept_id","emp_location"]
dept_columns = ["dept_id", "dept_name", "dept_head","dept_location"]

emp_df = spark.createDataFrame(emp_data,emp_columns)
dept_df = spark.createDataFrame(dept_data,dept_columns)

emp_df.display()
dept_df.display()

Inner Join with Filtering Columns and WHERE Condition

In [0]:
"""
inner_df = emp_df.join(dept_df, emp_df["emp_dept_id"]==dept_df["dept_id"], "inner")
inner_df.show()
where_df = inner_df.select("emp_id","emp_name","emp_salary","dept_name","dept_location").filter("emp_salary > 55000")
where_df.show()

"""

inner_where_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "inner")\
    .select("emp_id","emp_name","emp_salary","dept_name","dept_location")\
    .filter("emp_salary > 55000").show()

In [0]:
left_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "left").show()

In [0]:
left_anti_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "left_anti").show()

In [0]:
right_where_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "right")\
    .select("emp_id","emp_name","emp_salary","dept_name","dept_location")\
    .filter("emp_salary > 55000").show()

In [0]:
left_where_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "left")\
    .select("emp_id","emp_name","emp_salary","dept_name","dept_location")\
    .filter("emp_salary > 55000").show()

In [0]:
left_anti_where_df = emp_df.join(dept_df, emp_df["emp_dept_id"] == dept_df["dept_id"], "left_anti")\
    .select("emp_id","emp_name","emp_salary")\
    .filter("emp_salary > 55000").show()

### Practice Que
- Write a PySpark query to find employees whose location matches the location of their department.  
  Display emp_id, emp_name, emp_location, dept_name, and dept_location for matching records.

- Modify the code to find departments that have no employees assigned to them.  
  Display dept_id, dept_name, and dept_head.

- Write a PySpark query to get the average salary of employees in each department,  
  displaying dept_name and the calculated average_salary.

- List the employees who earn more than the average salary of their department.  
  Display emp_id, emp_name, emp_salary, dept_name, and dept_location.

In [0]:
#Write a PySpark query to find employees whose location matches the location of their department.Display emp_id, emp_name, emp_location, dept_name, and dept_location for matching records.

from pyspark.sql.functions import col
employee_locations_df = emp_df.join(dept_df,emp_df.emp_dept_id == dept_df.dept_id, "inner")\
    .filter(col("emp_location") == col("dept_location"))\
    .select("emp_id","emp_name","emp_location",col("dept_name").alias("depoartment"),col("dept_location").alias("Location"))
employee_locations_df.show()

In [0]:
#Modify the code to find departments that have no employees assigned to them. Display dept_id, dept_name, and dept_head
from pyspark.sql import functions as f

df = dept_df.join(emp_df,emp_df.emp_dept_id == dept_df.dept_id, "left_anti")\
    .select(f.col("dept_id"),f.col("dept_name"),f.col("dept_head"))
display(df)

In [0]:
#Write a PySpark query to get the average salary of employees in each department,displaying dept_name and the calculated average_salary.
from pyspark.sql import functions as f

avg_df = emp_df.groupBy("emp_dept_id")\
    .agg(f.avg("emp_salary").alias("Average_salary"))
result = avg_df.join(dept_df, avg_df.emp_dept_id==dept_df.dept_id, "right")\
    .select(f.col("dept_name").alias("Department"),f.col("Average_salary"))
result.show()

In [0]:
from pyspark.sql import functions as f

df3 = emp_df.join(dept_df, emp_df.emp_dept_id==dept_df.dept_id, "right")\
    .groupBy("dept_name")\
    .agg(f.avg("emp_salary").alias("Averge_salary"))
display(df3)

In [0]:
#List the employees who earn more than the average salary of their department.Display emp_id, emp_name, emp_salary, dept_name, and dept_location.
from pyspark.sql import functions as f

average_df = emp_df.groupBy(f.col("emp_dept_id").alias("department"))\
    .agg(f.avg("emp_salary").alias("Average_salary"))
average_df.show()

final_result = average_df.join(dept_df, average_df.department == dept_df.dept_id, "inner")\
    .join(emp_df, emp_df.emp_dept_id == dept_df.dept_id, "inner")\
    .filter(f.col("emp_salary")>f.col("Average_salary"))\
    .select(f.col("emp_id"),f.col("emp_name"),f.col("emp_salary"),f.col("dept_name"),f.col("dept_location"),f.col("department"))
display(final_result)

In [0]:
emp_df.write.mode("overwrite").saveAsTable("employees")
dept_df.write.mode("overwrite").saveAsTable("department")

In [0]:
%sql
--select * from department

select *from employees

In [0]:
%sql
--#List the employees who earn more than the average salary of their department.Display emp_id, emp_name, emp_salary, dept_name, and dept_location.
select e.emp_id,e.emp_name,e.emp_salary,d.dept_name,d.dept_location 
from employees e join department d 
on e.emp_dept_id = d.dept_id 
join (select emp_dept_id, avg(emp_salary) as average_salary from employees group by emp_dept_id ) a
on d.dept_id=a.emp_dept_id
where e.emp_salary> a.average_salary